## Training Toxicity Layer

In [ ]:
pip install 'accelerate>=1.1.0'

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt

# Confirm GPU is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

# Load processed toxicity data
df = pd.read_csv('../data/processed/processed_toxicity.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Device: NVIDIA GeForce RTX 5070
Shape: (160568, 7)
Columns: ['text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [3]:
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Train/val split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained('s-nlp/roberta_toxicity_classifier')

# Dataset class
class ToxicityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = torch.tensor(labels.values, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

# Build datasets
train_dataset = ToxicityDataset(train_df['text'], train_df[label_cols], tokenizer)
val_dataset = ToxicityDataset(val_df['text'], val_df[label_cols], tokenizer)

print("Datasets ready")

Train: 128454 | Val: 32114
Datasets ready


In [4]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"VRAM free: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 5070
VRAM total: 12.82 GB
VRAM free: 0.00 GB


In [5]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=6,
    problem_type='multi_label_classification'
)
model = model.to('cuda')
print("Model loaded on GPU")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1576.90it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on GPU


## Run 1 roberta-base

In [7]:
# Load model
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=6,
    problem_type='multi_label_classification'
)
model = model.to('cuda')
print("Model loaded on GPU")

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels.astype(int)
    f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    return {'f1_macro': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir='../models/roberta_toxicity',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_dir='../models/roberta_toxicity/logs',
    logging_steps=100,
    warmup_steps=500,
    weight_decay=0.01,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Ready to train")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2229.05it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be rem

Model loaded on GPU
Ready to train


In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.094433,0.046851,0.533491
2,0.070026,0.042659,0.626184
3,0.061423,0.044202,0.655400


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.77s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=12045, training_loss=0.08833350501846307, metrics={'train_runtime': 1842.6866, 'train_samples_per_second': 209.131, 'train_steps_per_second': 6.537, 'total_flos': 2.5349160995767296e+16, 'train_loss': 0.08833350501846307, 'epoch': 3.0})

In [9]:
from sklearn.metrics import classification_report
import torch
import numpy as np

# Get predictions on validation set
model.eval()
all_logits = []
all_labels = []

from torch.utils.data import DataLoader

val_loader = DataLoader(val_dataset, batch_size=32)

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels']
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        all_logits.append(logits.cpu())
        all_labels.append(labels)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

preds = (torch.sigmoid(all_logits) > 0.5).int().numpy()
labels_np = all_labels.int().numpy()

print(classification_report(
    labels_np, preds,
    target_names=label_cols,
    zero_division=0
))

               precision    recall  f1-score   support

        toxic       0.84      0.83      0.83      3178
 severe_toxic       0.50      0.42      0.46       315
      obscene       0.81      0.86      0.83      1763
       threat       0.42      0.53      0.46        80
       insult       0.76      0.78      0.77      1671
identity_hate       0.59      0.56      0.57       327

    micro avg       0.78      0.79      0.79      7334
    macro avg       0.65      0.66      0.66      7334
 weighted avg       0.78      0.79      0.79      7334
  samples avg       0.07      0.08      0.07      7334



## Run 2 roberta-base with class weights

In [10]:
import torch
import numpy as np

# Calculate class weights inversely proportional to frequency
label_counts = train_df[label_cols].sum()
total = len(train_df)
class_weights = torch.tensor(
    [total / (len(label_cols) * label_counts[col]) for col in label_cols],
    dtype=torch.float32
).to('cuda')

print("Class weights:")
for col, w in zip(label_cols, class_weights):
    print(f"  {col}: {w:.4f}")

Class weights:
  toxic: 1.7025
  severe_toxic: 16.7258
  obscene: 3.1549
  threat: 51.0955
  insult: 3.2656
  identity_hate: 17.6061


In [11]:
from torch import nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [13]:
# Reload best checkpoint from previous run
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    '../models/roberta_toxicity/checkpoint-8030',
    num_labels=6,
    problem_type='multi_label_classification'
)
model = model.to('cuda')
print("Loaded from checkpoint-8030 (best epoch 2)")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1431.64it/s]


Loaded from checkpoint-8030 (best epoch 2)


In [14]:
training_args = TrainingArguments(
    output_dir='../models/roberta_toxicity_weighted',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=100,
    warmup_steps=200,
    weight_decay=0.01,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Ready to train with class weights")

Ready to train with class weights


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.246335,0.150331,0.636098
2,0.091605,0.175377,0.658820


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.72s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=8030, training_loss=0.1824528689402275, metrics={'train_runtime': 1263.6786, 'train_samples_per_second': 203.302, 'train_steps_per_second': 6.354, 'total_flos': 1.6899440663844864e+16, 'train_loss': 0.1824528689402275, 'epoch': 2.0})

In [17]:
model = RobertaForSequenceClassification.from_pretrained(
    '../models/roberta_toxicity_weighted/checkpoint-8030',
    num_labels=6,
    problem_type='multi_label_classification'
)
model = model.to('cuda')
print("Loaded weighted model epoch 2")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1484.90it/s]


Loaded weighted model epoch 2


In [22]:
model.eval()
all_logits = []
all_labels = []

val_loader = DataLoader(val_dataset, batch_size=32)

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels']
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        all_logits.append(logits.cpu())
        all_labels.append(labels)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

preds = (torch.sigmoid(all_logits) > 0.5).int().numpy()
labels_np = all_labels.int().numpy()

print(classification_report(
    labels_np, preds,
    target_names=label_cols,
    zero_division=0
))

               precision    recall  f1-score   support

        toxic       0.78      0.85      0.82      3178
 severe_toxic       0.36      0.77      0.49       315
      obscene       0.77      0.88      0.82      1763
       threat       0.38      0.66      0.48        80
       insult       0.69      0.86      0.77      1671
identity_hate       0.54      0.61      0.57       327

    micro avg       0.71      0.84      0.77      7334
    macro avg       0.59      0.77      0.66      7334
 weighted avg       0.73      0.84      0.78      7334
  samples avg       0.07      0.08      0.07      7334



## s-nlp toxicity model with class weights

In [ ]:
from transformers import RobertaForSequenceClassification, RobertaConfig
import torch

# Load config from toxicity model but override num_labels
config = RobertaConfig.from_pretrained('s-nlp/roberta_toxicity_classifier')
config.num_labels = 6
config.problem_type = 'multi_label_classification'

# Load model with new config - encoder weights transfer, head reinitializes
model = RobertaForSequenceClassification.from_pretrained(
    's-nlp/roberta_toxicity_classifier',
    config=config,
    ignore_mismatched_sizes=True
)
model = model.to('cuda')
print("Loaded toxicity-aware RoBERTa with fresh 6-label head")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14052.79it/s]
RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([6, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([6])          

Notes:
- UNEXPECTED:	can be ignored when loading from different 

Loaded toxicity-aware RoBERTa with fresh 6-label head


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
    ~~~~~~~~~~~~~~~~^^^^^^

In [24]:
# Recalculate class weights
label_counts = train_df[label_cols].sum()
total = len(train_df)
class_weights = torch.tensor(
    [total / (len(label_cols) * label_counts[col]) for col in label_cols],
    dtype=torch.float32
).to('cuda')

# Training args
training_args = TrainingArguments(
    output_dir='../models/roberta_toxicity_v3',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=100,
    warmup_steps=500,
    weight_decay=0.01,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Ready to train")

Ready to train


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.328424,0.144648,0.607710
2,0.163378,0.123383,0.633959
3,0.206579,0.133922,0.661473


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.76s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=12045, training_loss=0.2600598484512779, metrics={'train_runtime': 1897.4807, 'train_samples_per_second': 203.091, 'train_steps_per_second': 6.348, 'total_flos': 2.5349160995767296e+16, 'train_loss': 0.2600598484512779, 'epoch': 3.0})

In [28]:
model.eval()
all_logits = []
all_labels = []

val_loader = DataLoader(val_dataset, batch_size=32)

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels']
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        all_logits.append(logits.cpu())
        all_labels.append(labels)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

preds = (torch.sigmoid(all_logits) > 0.5).int().numpy()
labels_np = all_labels.int().numpy()

print(classification_report(
    labels_np, preds,
    target_names=label_cols,
    zero_division=0
))

               precision    recall  f1-score   support

        toxic       0.80      0.86      0.83      3178
 severe_toxic       0.35      0.79      0.48       315
      obscene       0.74      0.91      0.82      1763
       threat       0.39      0.69      0.50        80
       insult       0.68      0.86      0.76      1671
identity_hate       0.47      0.75      0.58       327

    micro avg       0.70      0.86      0.77      7334
    macro avg       0.57      0.81      0.66      7334
 weighted avg       0.72      0.86      0.78      7334
  samples avg       0.07      0.08      0.07      7334



## Run 4 - Focal Loss, 5 Labels, Early Stopping (Final)

In [29]:
label_cols = ['toxic', 'obscene', 'threat', 'insult', 'identity_hate']

train_dataset = ToxicityDataset(train_df['text'], train_df[label_cols], tokenizer)
val_dataset = ToxicityDataset(val_df['text'], val_df[label_cols], tokenizer)

# Recalculate class weights for 5 labels
label_counts = train_df[label_cols].sum()
total = len(train_df)
class_weights = torch.tensor(
    [total / (len(label_cols) * label_counts[col]) for col in label_cols],
    dtype=torch.float32
).to('cuda')

print("Class weights:")
for col, w in zip(label_cols, class_weights):
    print(f"  {col}: {w:.4f}")

Class weights:
  toxic: 2.0430
  obscene: 3.7859
  threat: 61.3146
  insult: 3.9187
  identity_hate: 21.1273


In [31]:
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none'
        )
        pt = torch.exp(-bce)
        focal = ((1 - pt) ** self.gamma) * bce
        return focal.mean()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = FocalLoss(gamma=2.0, pos_weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
config = RobertaConfig.from_pretrained('s-nlp/roberta_toxicity_classifier')
config.num_labels = 5
config.problem_type = 'multi_label_classification'

model = RobertaForSequenceClassification.from_pretrained(
    's-nlp/roberta_toxicity_classifier',
    config=config,
    ignore_mismatched_sizes=True
)
model = model.to('cuda')
print("Loaded with 5 labels")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15740.97it/s]
RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([5])          
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([5, 768])

Notes:
- UNEXPECTED:	can be ignored when loading from different 

Loaded with 5 labels


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
    ~~~~~~~~~~~~~~~~^^^^^^

In [ ]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='../models/roberta_toxicity_v4',
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=True,
    learning_rate=2e-5,
    warmup_steps=1000,
    weight_decay=0.01,
    logging_steps=100,
)

# Reload fresh s-nlp model
model = RobertaForSequenceClassification.from_pretrained(
    's-nlp/roberta_toxicity_classifier',
    config=config,
    ignore_mismatched_sizes=True
)
model = model.to('cuda')

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Ready to train - Run 4 (focal loss, 5 labels, lr=2e-5, early stopping)")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12829.55it/s]
RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([5])          
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([5, 768])

Notes:
- UNEXPECTED:	can be ignored when loading from different 

Ready to train - Run 4 (focal loss, 5 labels, lr=2e-5, early stopping)


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/mnt/c/Users/DM77/youtube-comment-moderation/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
    ~~~~~~~~~~~~~~~~^^^^^^

In [37]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.167919,0.064191,0.594780
2,0.090344,0.069515,0.647769
3,0.094988,0.067208,0.648982
4,0.071897,0.085115,0.676568
5,0.056955,0.098668,0.696756
6,0.033691,0.107679,0.703170


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.70s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=24090, training_loss=0.08885489638389121, metrics={'train_runtime': 3820.2221, 'train_samples_per_second': 201.748, 'train_steps_per_second': 6.306, 'total_flos': 5.069786680810598e+16, 'train_loss': 0.08885489638389121, 'epoch': 6.0})

In [39]:
model.eval()
all_logits = []
all_labels = []

val_loader = DataLoader(val_dataset, batch_size=32)

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

preds = (torch.sigmoid(all_logits) > 0.45).int().numpy()
labels_np = all_labels.int().numpy()

print(classification_report(labels_np, preds, target_names=label_cols, zero_division=0))

               precision    recall  f1-score   support

        toxic       0.76      0.90      0.82      3178
      obscene       0.74      0.90      0.81      1763
       threat       0.41      0.74      0.52        80
       insult       0.68      0.85      0.76      1671
identity_hate       0.51      0.66      0.57       327

    micro avg       0.71      0.88      0.79      7019
    macro avg       0.62      0.81      0.70      7019
 weighted avg       0.72      0.88      0.79      7019
  samples avg       0.08      0.09      0.08      7019



### Moving Models to HuggingFace

In [41]:
model.push_to_hub('DavidMembreno/youtube-comment-moderation-roberta')
tokenizer.push_to_hub('DavidMembreno/youtube-comment-moderation-roberta')

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]
Processing Files (1 / 1): 100%|██████████|  499MB /  499MB, 1.50MB/s  
New Data Upload: 100%|██████████|  499MB /  499MB, 1.50MB/s  


CommitInfo(commit_url='https://huggingface.co/DavidMembreno/youtube-comment-moderation-roberta/commit/d071794df396745fa77e04bc02dc4d09a969bb83', commit_message='Upload tokenizer', commit_description='', oid='d071794df396745fa77e04bc02dc4d09a969bb83', pr_url=None, repo_url=RepoUrl('https://huggingface.co/DavidMembreno/youtube-comment-moderation-roberta', endpoint='https://huggingface.co', repo_type='model', repo_id='DavidMembreno/youtube-comment-moderation-roberta'), pr_revision=None, pr_num=None)

In [43]:
from huggingface_hub import HfApi
api = HfApi()

# Create repo first
api.create_repo(repo_id='DavidMembreno/youtube-comment-moderation-spam', repo_type='model', exist_ok=True)

# Then upload
api.upload_file(
    path_or_fileobj='../models/spam_classifier.pkl',
    path_in_repo='spam_classifier.pkl',
    repo_id='DavidMembreno/youtube-comment-moderation-spam',
    repo_type='model'
)

Processing Files (1 / 1): 100%|██████████| 1.43MB / 1.43MB,  549kB/s  
New Data Upload: 100%|██████████| 1.43MB / 1.43MB,  549kB/s  


CommitInfo(commit_url='https://huggingface.co/DavidMembreno/youtube-comment-moderation-spam/commit/185b19771f19c5f9ce80ab9ad0bc49f9b212bdcd', commit_message='Upload spam_classifier.pkl with huggingface_hub', commit_description='', oid='185b19771f19c5f9ce80ab9ad0bc49f9b212bdcd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/DavidMembreno/youtube-comment-moderation-spam', endpoint='https://huggingface.co', repo_type='model', repo_id='DavidMembreno/youtube-comment-moderation-spam'), pr_revision=None, pr_num=None)